# Hopenhayn Firm Dynamics Model: Deep Learning Solution

This notebook solves the Bellman equation for the stationary firm problem in Hopenhayn, Neira and Singhania (2022) using a neural network, mirroring the approach in the McCall search model notebook.

$$V(s) = \max\left\{0,\; \pi(s) + \beta \sum_{s'} P(s'|s)\, V(s')\right\}$$

where $\pi(s) = (p \cdot e^s \cdot \alpha^\alpha)^{1/(1-\alpha)} (1-\alpha) - c_f$ and $s$ follows a Tauchen-discretized AR(1).

## Importing packages

In [5]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from scipy.stats import norm
import time

## Config for plotting

In [6]:
plt.rcParams.update({
    # --- Text and fonts ---
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    # --- Figure dimensions ---
    "figure.figsize": (5.5, 3.8),
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
    # --- Axes and spines ---
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    # --- Ticks ---
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    # --- Grid ---
    "axes.grid": False,
    # --- Legend ---
    "legend.frameon": False,
    # --- Layout ---
    "figure.constrained_layout.use": True,
})

# Okabe-Ito colorblind-safe palette (same as HJB code)
COLOR_NN  = "#0072B2"    # Blue  â€” neural network
COLOR_REF = "#D55E00"    # Vermillion â€” VFI reference

## Model parameters

In [7]:
beta   = 1.0 / 1.04
alpha  = 0.64
pstar  = 1.0
wstar  = 1.0
rho    = 0.984150757243253
sigmaF = 0.245520815536363
muF    = -2.431373086987380
cf     = 2.298950284374937

## Tauchen discretization of the AR(1) process

In [8]:
def tauchen(mu, rho_val, sig, N, m=5):
    sigma_uncond = np.sqrt(sig**2 / (1 - rho_val**2))
    svec = np.linspace(mu - m * sigma_uncond, mu + m * sigma_uncond, N)
    step = svec[1] - svec[0]
    Pi = np.zeros((N, N))
    for j in range(N):
        for k in range(N):
            if k == 0:
                Pi[j, k] = norm.cdf((svec[0] - (1 - rho_val) * mu - rho_val * svec[j] + step / 2) / sig)
            elif k == N - 1:
                Pi[j, k] = 1.0 - norm.cdf((svec[-1] - (1 - rho_val) * mu - rho_val * svec[j] - step / 2) / sig)
            else:
                Pi[j, k] = (norm.cdf((svec[k] - (1 - rho_val) * mu - rho_val * svec[j] + step / 2) / sig)
                          - norm.cdf((svec[k] - (1 - rho_val) * mu - rho_val * svec[j] - step / 2) / sig))
    return svec, Pi

In [9]:
ns = 100
svec_np, Pi_np = tauchen(muF, rho, sigmaF, ns)
sMin, sMax = svec_np[0], svec_np[-1]
print(f"Grid: {ns} points, s in [{sMin:.2f}, {sMax:.2f}]")

Grid: 100 points, s in [-9.35, 4.49]


## Profit function

$$\pi(s) = \left(p \cdot e^s \cdot \alpha^\alpha\right)^{1/(1-\alpha)} (1-\alpha) - c_f$$

In [10]:
def profit_torch(s):
    return (pstar * torch.exp(s) * (alpha / wstar)**alpha) ** (1.0 / (1.0 - alpha)) * (1.0 - alpha) - cf

def profit_np(s):
    return (pstar * np.exp(s) * (alpha / wstar)**alpha) ** (1.0 / (1.0 - alpha)) * (1.0 - alpha) - cf

## Load VFI reference solution

Run `hopenhayn_VFI.ipynb` first to generate the CSVs.

In [17]:
ls

 El volumen de la unidad C no tiene etiqueta.
 El nï¿½mero de serie del volumen es: 6CD1-A07A

 Directorio de c:\Users\MVAZCAR\Dropbox\hopenhayn

24/03/2026  11:48    <DIR>          .
23/03/2026  19:26    <DIR>          ..
24/03/2026  11:48                24 .env
24/03/2026  11:48    <DIR>          .vscode
24/03/2026  00:37            23.605 deep_learning_hjb.py
24/03/2026  11:50    <DIR>          deepHopenhayn
23/03/2026  19:26    <DIR>          raw_code
23/03/2026  22:55    <DIR>          simple
               2 archivos         23.629 bytes
               6 dirs  893.383.991.296 bytes libres


In [16]:
import os

# Load VFI reference from CSVs (generated by hopenhayn_VFI.ipynb)
vfi_csv_v    = 'output_csv/v_VFI.csv'
vfi_csv_svec = 'output_csv/svec_VFI.csv'

V_vfi = np.genfromtxt(vfi_csv_v, delimiter=',')
svec_check = np.genfromtxt(vfi_csv_svec, delimiter=',')
assert np.allclose(svec_check, svec_np), "Grid mismatch between VFI and DL notebooks!"
print(f"Loaded VFI reference from {vfi_csv_v} ({len(V_vfi)} points)")

# Exit threshold
active_vfi = np.where(V_vfi > 0.01)[0]
sstar_vfi = svec_np[active_vfi[0]] if len(active_vfi) > 0 else svec_np[0]
print(f"VFI exit threshold: s* = {sstar_vfi:.4f}")

FileNotFoundError: v_VFI.csv not found.

## Plot: VFI value function

In [ ]:
plt.plot(svec_np, np.log1p(V_vfi), color=COLOR_REF, linewidth=1.5, label='VFI Value Function')
plt.xlabel(r'Productivity, $s$')
plt.ylabel(r'$\log(1 + V(s))$')
plt.legend()
plt.show()

## Defining the neural network

4 hidden layers Ã— 128 neurons, SiLU activation (smooth, better at kinks than ReLU/Tanh). Softplus output ensures V(s) â‰¥ 0.

In [ ]:
class NN(nn.Module):
    def __init__(self, dim_hidden=128, layers=4, hidden_bias=True):
        super().__init__()
        torch.manual_seed(123)
        module = []
        module.append(nn.Linear(1, dim_hidden, bias=hidden_bias))
        module.append(nn.SiLU())
        for i in range(layers - 1):
            module.append(nn.Linear(dim_hidden, dim_hidden, bias=hidden_bias))
            module.append(nn.SiLU())
        module.append(nn.Linear(dim_hidden, 1))
        module.append(nn.Softplus())  # ensures V(s) >= 0
        self.q = nn.Sequential(*module)

    def forward(self, x):
        return self.q(x)

## Defining the training data and data-loader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

svec_t = torch.tensor(svec_np, dtype=torch.float32, device=device).unsqueeze(1)  # (N, 1)
Pi_t   = torch.tensor(Pi_np, dtype=torch.float32, device=device)                 # (N, N)
pi_grid = profit_torch(svec_t)                                                    # (N, 1)

# No more DataLoader â€” we'll sample mini-batches instead
batchSize = 512

In [ ]:
num_epochs = 200001
print_epoch_frequency = 5000
tau = 0.01  # Polyak averaging rate for target network
early_stop_loss = 1e-6  # stop when running avg loss drops below this

## Initiating the neural network and setting up the optimizer

In [ ]:
import copy

v_hat = NN().to(device)

learning_rate = 1e-3
optimizer = torch.optim.Adam(v_hat.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Target network: Polyak-averaged copy for stable Bellman targets
target_net = copy.deepcopy(v_hat)
target_net.eval()
for p in target_net.parameters():
    p.requires_grad_(False)

## Expected value using the Tauchen transition matrix

$$E[V(s') | s_j] = \sum_k P(s_k | s_j) \cdot V(s_k)$$

In [ ]:
sigma_uncond = np.sqrt(sigmaF**2 / (1 - rho**2))
exit_center = 0.0  # adaptive â€” updated during training

def sample_s(batch_size):
    """
    Draw collocation points from a mixture:
      1/4 uniform on [sMin, sMax]
      1/4 normal around muF (ergodic region)
      1/4 normal around the exit region (lower tail)
      1/4 normal around the current exit threshold estimate
    """
    n1 = batch_size // 4
    n2 = batch_size // 4
    n3 = batch_size // 4
    n4 = batch_size - n1 - n2 - n3

    x1 = sMin + (sMax - sMin) * torch.rand(n1, 1, device=device)
    x2 = torch.normal(muF, sigma_uncond, size=(n2, 1)).clamp(sMin, sMax).to(device)
    x3 = torch.normal(muF - 2.0 * sigma_uncond, sigma_uncond, size=(n3, 1)).clamp(sMin, sMax).to(device)
    x4 = torch.normal(exit_center, 0.5, size=(n4, 1)).clamp(sMin, sMax).to(device)

    return torch.cat([x1, x2, x3, x4], dim=0)[torch.randperm(batch_size, device=device)]

def E_V_at_points(model_or_target, s_input, use_target=False):
    """
    Compute E[V(s')|s] for arbitrary points s by interpolating
    the Tauchen transition matrix rows.
    """
    # Evaluate V at all grid points
    if use_target:
        with torch.no_grad():
            V_grid = target_net(svec_t).squeeze()  # (N,)
    else:
        V_grid = model_or_target(svec_t).squeeze()  # (N,)

    # Interpolate Pi rows for arbitrary s
    s_flat = s_input.squeeze()  # (batch,)
    step = svec_t[1, 0] - svec_t[0, 0]
    idx_float = (s_flat - svec_t[0, 0]) / step
    idx_lo = idx_float.long().clamp(0, ns - 2)
    w_hi = (idx_float - idx_lo.float()).clamp(0.0, 1.0).unsqueeze(1)  # (batch, 1)

    Pi_interp = (1.0 - w_hi) * Pi_t[idx_lo] + w_hi * Pi_t[idx_lo + 1]  # (batch, N)
    EV = (Pi_interp * V_grid.unsqueeze(0)).sum(dim=1, keepdim=True)      # (batch, 1)
    return EV

def profit_at_points(s):
    """Profit at arbitrary s points."""
    return (pstar * torch.exp(s) * (alpha / wstar)**alpha) ** (1.0 / (1.0 - alpha)) * (1.0 - alpha) - cf

## Training loop

In [ ]:
# Collect snapshots for animation (including the mini-batch points)
snapshot_every = 500
snapshots = []
final_epoch = num_epochs

t_start = time.time()

# Track running average loss for early stopping
loss_avg = 1.0

for epoch in range(num_epochs):
    # Sample mini-batch (concentrated near exit threshold)
    s_input = sample_s(batchSize)
    
    optimizer.zero_grad()

    # LHS: V_hat(s) â€” always >= 0 thanks to Softplus
    lhs_v = v_hat(s_input)

    # RHS uses TARGET network for stable E[V'] targets
    EV = E_V_at_points(None, s_input, use_target=True)
    pi = profit_at_points(s_input)
    rhs_v = torch.max(torch.zeros_like(s_input), pi + beta * EV)

    # Bellman residual in log-space
    residual = torch.log1p(lhs_v) - torch.log1p(rhs_v)
    loss = (residual ** 2).mean()

    loss.backward()
    optimizer.step()
    scheduler.step()

    # Polyak update of target network
    with torch.no_grad():
        for p, p_tgt in zip(v_hat.parameters(), target_net.parameters()):
            p_tgt.data.mul_(1.0 - tau).add_(p.data * tau)

    # Running average for early stopping
    loss_avg = 0.99 * loss_avg + 0.01 * loss.item()

    # Snapshot: save V on grid AND the mini-batch points used this epoch
    if epoch % snapshot_every == 0:
        with torch.no_grad():
            V_snap = v_hat(svec_t).cpu().numpy().flatten()
            # Evaluate NN at the mini-batch points for plotting
            V_at_batch = v_hat(s_input).cpu().numpy().flatten()
        s_batch_np = s_input.cpu().numpy().flatten()
        snapshots.append((epoch, V_snap.copy(), loss.item(), s_batch_np.copy(), V_at_batch.copy()))

    if epoch % print_epoch_frequency == 0:
        print(f"epoch = {epoch:>6d}, loss = {loss.item():.3e}, avg = {loss_avg:.3e}")
        # Update adaptive exit center
        with torch.no_grad():
            V_test = v_hat(svec_t).squeeze()
            pos_idx = (V_test > 0.01).nonzero()
            if len(pos_idx) > 0:
                exit_center = svec_t[pos_idx[0]].item()

    # Early stopping
    if epoch > 10000 and loss_avg < early_stop_loss:
        print(f"\n*** Early stopping at epoch {epoch} (avg loss = {loss_avg:.3e}) ***")
        final_epoch = epoch
        break

t_end = time.time()
dl_time = t_end - t_start
print(f"\nDL iteration time: {dl_time:.2f} seconds ({final_epoch} epochs)")

## Evaluation

In [ ]:
with torch.no_grad():
    V_nn = v_hat(svec_t).cpu().numpy().flatten()

# Add final snapshot (no batch points for the final static one)
s_final_batch = sample_s(batchSize).cpu().numpy().flatten()
with torch.no_grad():
    V_final_batch = v_hat(torch.tensor(s_final_batch, dtype=torch.float32, device=device).unsqueeze(1)).cpu().numpy().flatten()
snapshots.append((final_epoch, V_nn.copy(), loss.item(), s_final_batch, V_final_batch))

In [ ]:
# Relative errors
relative_error = np.where(V_vfi > 0.01, (V_vfi - V_nn) / V_vfi, 0.0)
abs_rel_error = np.abs(relative_error)

# Exit thresholds
active_nn = np.where(V_nn > 0.01)[0]
sstar_nn = svec_np[active_nn[0]] if len(active_nn) > 0 else svec_np[0]

print(f"Exit threshold â€” VFI:  s* = {sstar_vfi:.4f}")
print(f"Exit threshold â€” NN:   s* = {sstar_nn:.4f}")
print(f"Difference:            {abs(sstar_nn - sstar_vfi):.4f}")
print(f"\nMax rel error (V>1):   {np.max(abs_rel_error[V_vfi > 1.0])*100:.2f}%")
print(f"Mean rel error (V>1):  {np.mean(abs_rel_error[V_vfi > 1.0])*100:.2f}%")
print(f"\nDL time:   {dl_time:.2f} s")

In [ ]:
# Save outputs for external plotting
np.savetxt('output_csv/v_nn_DL.csv', V_nn, delimiter=',')
np.savetxt('output_csv/svec_nn_DL.csv', svec_np, delimiter=',')
print("Saved v_nn_DL.csv, svec_nn_DL.csv")

## Mini-batch sampling visualization

The key difference from the McCall notebook (which uses full-batch on a uniform grid) is that we sample collocation points from a **mixture distribution** that concentrates near the exit threshold $s^*$.

- **Uniform** (gray): global coverage across $[s_{\min}, s_{\max}]$
- **Ergodic** (blue): centered at $\mu_F$ â€” where firms spend most time
- **Lower tail** (orange): centered at $\mu_F - 2\sigma$ â€” near the exit region
- **Exit-focused** (red): centered at the current $\hat{s}^*$ estimate â€” where the kink is

This ensures the NN gets informative gradients at the hardest part of the function (the kink) rather than wasting signal on the flat $V=0$ region.

In [ ]:
# Visualize the sampling distribution vs the value function
fig, ax1 = plt.subplots(figsize=(10, 5))

# Draw a large sample to show the density
n_show = 50000
n1 = n_show // 4; n2 = n_show // 4; n3 = n_show // 4; n4 = n_show - n1 - n2 - n3
np.random.seed(42)
x_uniform  = np.random.uniform(sMin, sMax, n1)
x_ergodic  = np.clip(np.random.normal(muF, sigma_uncond, n2), sMin, sMax)
x_lower    = np.clip(np.random.normal(muF - 2*sigma_uncond, sigma_uncond, n3), sMin, sMax)
x_exit     = np.clip(np.random.normal(exit_center, 0.5, n4), sMin, sMax)

bins = np.linspace(sMin, sMax, 80)

# Stacked histogram of the 4 components
ax1.hist([x_uniform, x_ergodic, x_lower, x_exit], bins=bins, stacked=True,
         color=["#CCCCCC", "#4393C3", "#F4A582", "#D6604D"],
         alpha=0.7, label=["Uniform", rf"Ergodic ($\mu_F$={muF:.1f})",
                           rf"Lower tail ($\mu_F-2\sigma$)", rf"Exit-focused ($s^*$={exit_center:.2f})"],
         density=True)
ax1.set_xlabel(r"Productivity: $s$")
ax1.set_ylabel("Sampling density")
ax1.axvline(x=sstar_vfi, color="k", linestyle="--", linewidth=1.5, label=rf"$s^*={sstar_vfi:.3f}$")

# Overlay the value function on a secondary axis
ax2 = ax1.twinx()
ax2.plot(svec_np, V_vfi, color="k", linewidth=2, alpha=0.5)
ax2.set_ylabel(r"$V(s)$", color="k", alpha=0.5)
ax2.tick_params(axis='y', colors='gray')

ax1.set_title("Mini-batch sampling: where collocation points are drawn")
ax1.legend(loc="upper right", fontsize=10)
plt.tight_layout()
plt.savefig("output_figures/sampling_distribution_DL.pdf")
plt.savefig("output_figures/sampling_distribution_DL.png", dpi=300)
plt.show()
print("Saved sampling_distribution_DL.pdf/png")

In [ ]:
# Exit region boundaries
exit_s = sstar_vfi
active_mask = V_vfi > 0.01
well_active = V_vfi > 1.0  # exclude near-zero V for mean
s_active_min = svec_np[active_mask][0] - 0.5
s_active_max = sMax + 0.2

fig, axes = plt.subplots(1, 2, figsize=(5.5, 2.8))

# ---- Panel (a): Value function in log scale ----
log_V_vfi = np.log1p(V_vfi)
log_V_nn  = np.log1p(V_nn)

ax = axes[0]
ax.plot(svec_np, log_V_vfi, color=COLOR_REF, linewidth=1.5, label='VFI')
ax.plot(svec_np, log_V_nn, color=COLOR_NN, linewidth=1.5, linestyle='--', label='Neural network')
y_max_log = max(log_V_vfi.max(), log_V_nn.max())
ax.set_ylim(-0.3, y_max_log * 1.05)
ax.set_xlim(sMin - 0.2, sMax + 0.2)
ax.axvspan(sMin - 0.2, exit_s, color="gray", alpha=0.15)
ax.set_xlabel(r'Productivity, $s$')
ax.set_ylabel(r'$\log(1 + V(s))$')
ax.set_title(r'(a) Value function')
ax.legend(loc='upper left')

# ---- Panel (b): Relative error in %, with threshold lines ----
ax = axes[1]
err_pct = abs_rel_error[active_mask] * 100  # convert to %
ax.plot(svec_np[active_mask], err_pct, color=COLOR_NN, linewidth=1.0)
ax.set_yscale("log")
ax.set_xlim(s_active_min, s_active_max)
ax.set_ylim(1e-3, 500)
# Threshold lines
ax.axhline(y=10, color='gray', linestyle=':', linewidth=0.7, alpha=0.7)
ax.axhline(y=1, color='gray', linestyle=':', linewidth=0.7, alpha=0.7)
ax.text(s_active_max - 0.1, 12, '10%', ha='right', fontsize=8, color='gray')
ax.text(s_active_max - 0.1, 1.2, '1%', ha='right', fontsize=8, color='gray')
# Exit threshold
ax.axvline(x=exit_s, color=COLOR_REF, linestyle='--', linewidth=0.8,
           label=rf'$s^*={exit_s:.3f}$')
# Mean error annotation
mean_err = np.mean(abs_rel_error[well_active]) * 100
ax.text(0.95, 0.05, f'Mean error (interior): {mean_err:.1f}%',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8,
        bbox=dict(facecolor='white', edgecolor='gray', alpha=0.8, boxstyle='round,pad=0.3'))
ax.set_xlabel(r'Productivity, $s$')
ax.set_ylabel(r'Relative error (%)')
ax.set_title(r'(b) Approximation error')
ax.legend(fontsize=8)

fig.savefig("output_figures/value_function_comparison_DL.pdf")
fig.savefig("output_figures/value_function_comparison_DL.png", dpi=300)
plt.show()
print("Saved value_function_comparison_DL.pdf/png")